# LAB 08 — Pretrained Transformer Fine-tuning

Notebook này chạy Application và Experiment 1–5 của LAB 08 trên IMDb. Mặc định dùng `distilbert-base-uncased` để phù hợp GPU Kaggle; có thể đổi sang `bert-base-uncased` nếu đủ tài nguyên.

Mỗi lần huấn luyện đều lưu lịch sử train/validation loss và vẽ đồ thị loss. Các kết quả cuối được lưu vào `results.csv`.

## 0. Kaggle setup

Trong Kaggle:

1. Tạo Notebook mới và bật **GPU** tại `Settings > Accelerator > GPU`.
2. Bật Internet tại `Settings > Internet` để clone repository và tải model/dataset.
3. Chạy cell dưới đây. Repository được clone vào `/kaggle/working/NLP`.

Nếu repository là private, thay URL bằng URL có quyền truy cập hoặc upload repository như Kaggle Dataset.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Tralaiday/NLP.git"
REPO_BRANCH = "lab8-pretrained-classification"
REPO_DIR = Path("/kaggle/working/NLP")

if not (REPO_DIR / "Lab8").exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Lab8 exists:", (REPO_DIR / "Lab8").exists())

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "transformers": "transformers",
    "datasets": "datasets",
    "seaborn": "seaborn",
}
for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True,
        )

import os
import random
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer
from Lab8.pretrained_classification import (
    IMDbDataset,
    PretrainedClassifier,
    count_parameters,
    evaluate,
    fit,
    load_imdb,
    set_seed,
    split_imdb,
)

OUTPUT_DIR = REPO_DIR / "Lab8"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = OUTPUT_DIR / "data" / "IMDB Dataset.csv"
MODEL_NAME = "distilbert-base-uncased"
SEED = 2026
BATCH_SIZE = 16
EPOCHS = 1
LEARNING_RATE = 2e-5
NUM_WORKERS = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Dataset, tokenizer và các hàm dùng chung

Tất cả experiment dùng cùng train/validation/test split. Validation dùng để theo dõi trong quá trình train; test chỉ dùng cho đánh giá cuối. Có thể đặt `MAX_TRAIN_SAMPLES` nhỏ hơn `None` để chạy thử nhanh trên Kaggle, nhưng báo cáo chính nên chạy với toàn bộ training split.

In [ ]:
MAX_TRAIN_SAMPLES = None
data = load_imdb(str(DATA_PATH))
train_df, validation_df, test_df = split_imdb(data, seed=SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(
        n=min(MAX_TRAIN_SAMPLES, len(train_df)),
        random_state=SEED,
    ).reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Tokenizer:", MODEL_NAME)

In [ ]:
results = []

def make_loader(frame, max_length, shuffle):
    dataset = IMDbDataset(frame, tokenizer, max_length=max_length)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

def new_pretrained(freeze_encoder=False):
    return PretrainedClassifier(
        model_name=MODEL_NAME,
        freeze_encoder=freeze_encoder,
    ).to(device)

class ScratchClassifier(nn.Module):
    """Small randomly initialized Transformer encoder for Experiment 1."""

    def __init__(self, vocab_size, max_length, hidden_size=128, heads=4, layers=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(max_length, hidden_size)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=heads,
            dim_feedforward=hidden_size * 2,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.classifier = nn.Linear(hidden_size, 2)
        self.max_length = max_length

    def forward(self, input_ids, attention_mask):
        positions = torch.arange(
            input_ids.size(1), device=input_ids.device
        ).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.encoder(x, src_key_padding_mask=~attention_mask.bool())
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(pooled)

def train_and_test(model, train_frame, validation_frame, test_frame, max_length):
    train_loader = make_loader(train_frame, max_length, shuffle=True)
    validation_loader = make_loader(validation_frame, max_length, shuffle=False)
    test_loader = make_loader(test_frame, max_length, shuffle=False)
    training = fit(
        model,
        train_loader,
        validation_loader,
        device,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
    )
    test_metrics = evaluate(model, test_loader, device)
    return model, training, test_metrics

def add_result(name, model, metrics, training, max_length, train_size):
    results.append({
        "experiment": name,
        "model": MODEL_NAME,
        "train_size": train_size,
        "max_length": max_length,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "training_time_seconds": training["training_time_seconds"],
        "parameters": count_parameters(model),
        "trainable_parameters": count_parameters(model, trainable_only=True),
    })

def plot_loss(histories, title, filename):
    plt.figure(figsize=(7, 4))
    for label, history in histories.items():
        epochs = [item["epoch"] for item in history]
        train_loss = [item["train_loss"] for item in history]
        validation_loss = [item["validation_loss"] for item in history]
        plt.plot(epochs, train_loss, marker="o", label=f"{label} train")
        plt.plot(epochs, validation_loss, marker="x", linestyle="--", label=f"{label} validation")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=150)
    plt.show()
    return path

## 2. Application — IMDb sentiment classification

Pipeline: review → tokenizer → `input_ids`/`attention_mask` → pretrained Transformer → `[CLS]` representation → linear classification head → logits → positive/negative.

In [ ]:
MAX_LENGTH = 128
application_model, application_training, application_metrics = train_and_test(
    new_pretrained(freeze_encoder=False),
    train_df,
    validation_df,
    test_df,
    MAX_LENGTH,
)
add_result(
    "Application - pretrained fine-tuning",
    application_model,
    application_metrics,
    application_training,
    MAX_LENGTH,
    len(train_df),
)
plot_loss(
    {"pretrained": application_training["history"]},
    "Application training loss",
    "loss_application.png",
)
print({key: application_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})

### Application evaluation: confusion matrix and confidence

In [ ]:
cm = confusion_matrix(application_metrics["labels"], application_metrics["predictions"])
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("IMDb confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

confidence_df = pd.DataFrame({
    "true": application_metrics["labels"],
    "prediction": application_metrics["predictions"],
    "confidence": application_metrics["probabilities"],
})
confidence_df["correct"] = confidence_df["true"] == confidence_df["prediction"]
confidence_df["confidence_group"] = pd.cut(
    confidence_df["confidence"],
    bins=[0.0, 0.6, 0.8, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
confidence_summary = confidence_df.groupby("confidence_group", observed=False)["correct"].mean()
confidence_summary.plot(kind="bar", ylim=(0, 1), ylabel="Accuracy", title="Confidence vs correctness")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confidence_analysis.png", dpi=150)
plt.show()
confidence_summary

## 3. Experiment 1 — Pretrained vs From Scratch

Giữ nguyên dataset, split, metric, maximum sequence length và số epoch. Model from scratch dùng cùng tokenizer nhưng Transformer encoder được khởi tạo ngẫu nhiên.

In [ ]:
scratch_model = ScratchClassifier(
    tokenizer.vocab_size,
    max_length=MAX_LENGTH,
).to(device)
scratch_model, scratch_training, scratch_metrics = train_and_test(
    scratch_model,
    train_df,
    validation_df,
    test_df,
    MAX_LENGTH,
)
add_result(
    "Experiment 1 - from scratch",
    scratch_model,
    scratch_metrics,
    scratch_training,
    MAX_LENGTH,
    len(train_df),
)
plot_loss(
    {
        "from scratch": scratch_training["history"],
        "pretrained": application_training["history"],
    },
    "Experiment 1 loss comparison",
    "loss_experiment_1.png",
)
pd.DataFrame(results).query("experiment.str.contains('Application|Experiment 1')", engine="python")[["experiment", "accuracy", "f1", "training_time_seconds", "parameters"]]

## 4. Experiment 2 — Frozen encoder vs Fine-tuning

Ở chế độ frozen, encoder không cập nhật tham số và chỉ classification head được train. Ở chế độ fine-tuning, encoder và head đều train.

In [ ]:
frozen_model, frozen_training, frozen_metrics = train_and_test(
    new_pretrained(freeze_encoder=True),
    train_df,
    validation_df,
    test_df,
    MAX_LENGTH,
)
fine_tuned_model, fine_tuned_training, fine_tuned_metrics = train_and_test(
    new_pretrained(freeze_encoder=False),
    train_df,
    validation_df,
    test_df,
    MAX_LENGTH,
)
add_result("Experiment 2 - frozen", frozen_model, frozen_metrics, frozen_training, MAX_LENGTH, len(train_df))
add_result("Experiment 2 - fine-tuning", fine_tuned_model, fine_tuned_metrics, fine_tuned_training, MAX_LENGTH, len(train_df))
plot_loss(
    {"frozen": frozen_training["history"], "fine-tuning": fine_tuned_training["history"]},
    "Experiment 2 loss comparison",
    "loss_experiment_2.png",
)
pd.DataFrame(results).query("experiment.str.startswith('Experiment 2')", engine="python")[["experiment", "accuracy", "f1", "trainable_parameters"]]

## 5. Experiment 3 — Amount of training data

Validation và test giữ cố định. Chỉ training subset thay đổi theo các mức 10%, 25%, 50% và 100%.

In [ ]:
data_histories = {}
data_rows = []
for fraction in (0.10, 0.25, 0.50, 1.00):
    subset = train_df.sample(frac=fraction, random_state=SEED).reset_index(drop=True)
    model, training, metrics = train_and_test(
        new_pretrained(freeze_encoder=False),
        subset,
        validation_df,
        test_df,
        MAX_LENGTH,
    )
    label = f"{int(fraction * 100)}%"
    data_histories[label] = training["history"]
    add_result(f"Experiment 3 - {label} training data", model, metrics, training, MAX_LENGTH, len(subset))
    data_rows.append({"training_data": label, "accuracy": metrics["accuracy"], "f1": metrics["f1"]})

plot_loss(data_histories, "Experiment 3 loss comparison", "loss_experiment_3.png")
data_efficiency = pd.DataFrame(data_rows)
data_efficiency.to_csv(OUTPUT_DIR / "data_efficiency.csv", index=False)
plt.figure(figsize=(6, 4))
plt.plot(data_efficiency["training_data"], data_efficiency["accuracy"], marker="o", label="Accuracy")
plt.plot(data_efficiency["training_data"], data_efficiency["f1"], marker="x", label="F1")
plt.xlabel("Training data")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Data-efficiency curve")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "data_efficiency.png", dpi=150)
plt.show()
data_efficiency

## 6. Experiment 4 — Maximum sequence length

So sánh `max_length` bằng 64, 128 và 256. Khi tăng độ dài, chi phí self-attention tăng xấp xỉ theo `O(T²)`.

In [ ]:
length_histories = {}
for max_length in (64, 128, 256):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model, training, metrics = train_and_test(
        new_pretrained(freeze_encoder=False),
        train_df,
        validation_df,
        test_df,
        max_length,
    )
    peak_memory_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else np.nan
    )
    add_result(
        f"Experiment 4 - max length {max_length}",
        model,
        metrics,
        training,
        max_length,
        len(train_df),
    )
    results[-1]["gpu_memory_mb"] = peak_memory_mb
    length_histories[str(max_length)] = training["history"]

plot_loss(length_histories, "Experiment 4 loss comparison", "loss_experiment_4.png")
pd.DataFrame(results).query("experiment.str.startswith('Experiment 4')", engine="python")[["experiment", "max_length", "accuracy", "f1", "training_time_seconds", "gpu_memory_mb"]]

## 7. Experiment 5 — Domain shift

Model được train trên IMDb (movie reviews) và đánh giá trên IMDb test cùng một dataset khác domain là Rotten Tomatoes. Bật Internet trên Kaggle để tải `rotten_tomatoes` từ Hugging Face Datasets.

In [ ]:
from datasets import load_dataset

rotten = load_dataset("rotten_tomatoes")
rotten_test = pd.DataFrame({
    "review": rotten["test"]["text"],
    "label": rotten["test"]["label"],
})
rotten_test["sentiment"] = rotten_test["label"].map({0: "negative", 1: "positive"})

rotten_loader = make_loader(rotten_test, MAX_LENGTH, shuffle=False)
imdb_domain_metrics = evaluate(application_model, make_loader(test_df, MAX_LENGTH, False), device)
rotten_domain_metrics = evaluate(application_model, rotten_loader, device)
results.extend([
    {
        "experiment": "Experiment 5 - in domain IMDb",
        "model": MODEL_NAME,
        "train_size": len(train_df),
        "max_length": MAX_LENGTH,
        "accuracy": imdb_domain_metrics["accuracy"],
        "precision": imdb_domain_metrics["precision"],
        "recall": imdb_domain_metrics["recall"],
        "f1": imdb_domain_metrics["f1"],
    },
    {
        "experiment": "Experiment 5 - out of domain Rotten Tomatoes",
        "model": MODEL_NAME,
        "train_size": len(train_df),
        "max_length": MAX_LENGTH,
        "accuracy": rotten_domain_metrics["accuracy"],
        "precision": rotten_domain_metrics["precision"],
        "recall": rotten_domain_metrics["recall"],
        "f1": rotten_domain_metrics["f1"],
    },
])
plot_loss({"IMDb training": application_training["history"]}, "Experiment 5 training loss", "loss_experiment_5.png")
pd.DataFrame(results).query("experiment.str.startswith('Experiment 5')", engine="python")[["experiment", "accuracy", "f1"]]

## 8. Lưu kết quả

`results.csv` chứa accuracy, precision, recall, F1, training time, số parameters và các thông số chính để báo cáo.

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
results_df

## 9. Đưa notebook và kết quả lên GitHub

Sau khi chạy xong và kiểm tra các output, chạy các lệnh sau trong Kaggle Terminal hoặc một code cell có tiền tố `!`:

```bash
cd /kaggle/working/NLP
git config user.name "Your Name"
git config user.email "your-email@example.com"
git add Lab8/fine_tuning.ipynb Lab8/results.csv Lab8/*.png Lab8/data_efficiency.csv
git commit -m "Add Lab8 fine-tuning experiments"
git push origin lab8-pretrained-classification
```

Nếu GitHub yêu cầu xác thực, dùng GitHub token có quyền ghi repository hoặc push từ máy cá nhân sau khi tải các file kết quả từ Kaggle. Không ghi token trực tiếp vào notebook.